In [1]:
import cutlass
import cutlass.cute as cute

@cute.jit
def foo(x: cutlass.Int32, y: cutlass.Constexpr):
    print("x = ", x)        # Prints x = ?
    print("y = ", y)        # Prints y = 2
    cute.printf("x: {}", x) # Prints x: 2
    cute.printf("y: {}", y) # Prints y: 2

foo(2, 2)

x =  ?
y =  2
x: 2
y: 2


In [8]:
import torch
import cutlass
from cutlass.cute.runtime import from_dlpack

@cute.jit
def foo(tensor):
    print(f"tensor.layout: {tensor.layout}")  # Prints tensor layout at compile time
    cute.printf("tensor: {}", tensor)         # Prints tensor values at runtime


a = torch.tensor([1, 2, 3], dtype=torch.uint16)
a_pack = from_dlpack(a)
compiled_func = cute.compile(foo, a_pack)
compiled_func(a_pack)

tensor.layout: (3):(1)


0

tensor: raw_ptr(0x000000003852bc40: i16, generic, align<2>) o (3):(1) = 
  ( 1, 2, 3 )


In [15]:
print(compiled_func.ir_module)

module attributes {gpu.container_module} {
  llvm.func @_cudaDeviceGetAttribute(!llvm.ptr, i32, i32) -> i32
  llvm.func @_cudaGetDevice(!llvm.ptr) -> i32
  llvm.func @_cudaKernelSetAttributeForDevice(!llvm.ptr, i32, i32, i32) -> i32
  llvm.func @_cudaFuncSetAttribute(!llvm.ptr, i32, i32) -> i32
  llvm.func @_cudaLibraryGetKernel(!llvm.ptr, !llvm.ptr, !llvm.ptr) -> i32
  llvm.func @_cudaLibraryLoadData(!llvm.ptr, !llvm.ptr, !llvm.ptr, !llvm.ptr, i32, !llvm.ptr, !llvm.ptr, i32) -> i32
  llvm.func @printf(!llvm.ptr, ...) -> i32
  llvm.mlir.global internal constant @printfFormat_0("tensor: raw_ptr(0x%016llx: i16, generic, align<2>) o (3):(1) = \0A  ( %d, %d, %d )\0A\00") {addr_space = 0 : i32}
  llvm.mlir.global internal constant @kernels_binary("P\EDU\BA\01\00\10\00(\08\00\00\00\00\00\00\02\00\01\01h\00\00\00\C0\07\00\00\00\00\00\00\00\00\00\00H\00\00\00\08\00\01\00y\00\00\00@\00\00\00\07\00\00\00\11\00\10\01\00\00\00\00\00\00\00\00\00\00\00\00\00\00\00\00\00\00\00\00kernels\00P\00\00\00\

In [16]:
b = torch.tensor([11, 12, 13, 14, 15], dtype=torch.uint16)
b_pack = from_dlpack(b)
compiled_func(b_pack)  # ❌ This results in an unexpected result at runtime due to type mismatch

0

tensor: raw_ptr(0x000000002a657640: i16, generic, align<2>) o (3):(1) = 
  ( 11, 12, 13 )


In [20]:
import torch
import cutlass
from cutlass.cute.runtime import from_dlpack

@cute.jit
def foo(tensor):
    print(tensor.layout)  # Prints (?,?):(?,1) for dynamic layout

a = torch.tensor([[1, 2], [3, 4]], dtype=torch.uint16)
compiled_func = cute.compile(foo, a)
compiled_func(a)

b = torch.tensor([[11, 12], [13, 14], [15, 16]], dtype=torch.uint16)
compiled_func(b)  # Reuse the same compiled function for different shape

(?,?):(?,1)


0

In [21]:
# Global variable
a = 1

@cute.jit
def add(b):
   return a + b

# Cache is empty at beginning

# First call: cache miss triggers compilation
result = add(2) # result = 3
# Cache now has one instance

# Second call: cache hit reuses cached JIT Executor
result = add(2) # result = 3

a = 2
# Third call: cache miss due to changed IR code triggers recompilation
result = add(2) # result = 4
# Cache now has two instances

In [22]:
import torch
import cutlass.cute as cute

@cute.jit
def foo(src):
    """
    The following lines print

    ptr<f32, generic> o (?,?,?):(?,?,1)
    <class 'cutlass.cute.core._Tensor'>
    """
    print(src)
    print(type(src))

a = torch.randn(30, 20, 32, device="cpu")
foo(a)

tensor<ptr<f32, generic> o (?,?,?):(?,?,1)>
<class 'cutlass.cute.tensor._Tensor'>


In [23]:
import torch
import cutlass
from cutlass.cute.runtime import from_dlpack

x = torch.randn(30, 20, device="cpu")
y = from_dlpack(x)

print(y.shape)        # (30, 20)
print(y.stride)       # (20, 1)
print(y.memspace)     # generic (if torch tensor in on device memory, memspace will be gmem)
print(y.element_type) # Float32
print(y)              # Tensor<0x000000000875f580@generic o (30, 20):(20, 1)>

(30, 20)
(20, 1)
0
Float32
Tensor<0x000000003a1e7a80@generic o (30,20):(20,1)>


In [26]:
import torch
from cutlass.cute.runtime import from_dlpack

# (8,4,16,2):(2,16,64,1)
a = torch.randn(16, 4, 8, 2).permute(2, 1, 0, 3)
# (1,4,1,32,1):(4,1,4,4,4) => torch tensor when dimension has shape 1, its stride is degenerated to 1,
# resulting in (1,4,1,32,1):(1,1,1,4,1)
b = torch.randn(32, 1, 1, 1, 4).permute(3, 4, 1, 0, 2)
# (2,2):(8,2)
c = torch.randn(3, 4)[::2, ::2]
# (3,1,1,5):(5,0,0,1)
d = torch.randn(3, 1, 1, 5).expand(3, 4, 2, 5)

In [35]:
for tensor in [a, b, c, d]:
    print("Tensor Shape: ", tensor.shape)
    print("Tensor Stride:", tensor.stride())
    cute_tensor = from_dlpack(tensor)
    print(cute_tensor)
    print("Cute Tensor shape: ", cute_tensor.shape)
    print("Cute Tensor stride:", cute_tensor.stride)
    print("-----")


Tensor Shape:  torch.Size([8, 4, 16, 2])
Tensor Stride: (2, 16, 64, 1)
Tensor<0x000000003a262a40@generic o (8,4,16,2):(2,16,64,1)>
Cute Tensor shape:  (8, 4, 16, 2)
Cute Tensor stride: (2, 16, 64, 1)
-----
Tensor Shape:  torch.Size([1, 4, 1, 32, 1])
Tensor Stride: (4, 1, 4, 4, 4)
Tensor<0x000000003ac62e40@generic o (1,4,1,32,1):(1,1,1,4,1)>
Cute Tensor shape:  (1, 4, 1, 32, 1)
Cute Tensor stride: (1, 1, 1, 4, 1)
-----
Tensor Shape:  torch.Size([2, 2])
Tensor Stride: (8, 2)
Tensor<0x000000003a1e87c0@generic o (2,2):(8,2)>
Cute Tensor shape:  (2, 2)
Cute Tensor stride: (8, 2)
-----
Tensor Shape:  torch.Size([3, 4, 2, 5])
Tensor Stride: (5, 0, 0, 1)
Tensor<0x000000003a328ec0@generic o (3,4,2,5):(5,0,0,1)>
Cute Tensor shape:  (3, 4, 2, 5)
Cute Tensor stride: (5, 0, 0, 1)
-----


In [34]:
cute_tensor.stride


(2, 16, 64, 1)